# 02 Data Cleaning

Transform raw NYC restaurant inspection data into clean, analysis-ready tables.
Each cleaning step is written as a function so it can be reused in the pipeline later.

In [1]:
import pandas as pd

RAW_PATH = "../data/raw/restaurant_inspections_raw.csv"
PROCESSED_DIR = "../data/processed/"

raw = pd.read_csv(RAW_PATH, dtype=str)
print(raw.shape)

(294406, 27)


### Step 1: Standardize column names
Convert column names to lowercase snake_case (e.g., `INSPECTION DATE` → `inspection_date`) so they work cleanly in both Python and SQL.

In [2]:
def standardize_columns(df):
    """Convert column names to lowercase snake_case."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


df_named = standardize_columns(raw)
print(df_named.columns.tolist())

['camis', 'dba', 'boro', 'building', 'street', 'zipcode', 'phone', 'cuisine_description', 'inspection_date', 'action', 'violation_code', 'violation_description', 'critical_flag', 'score', 'grade', 'grade_date', 'record_date', 'inspection_type', 'latitude', 'longitude', 'community_board', 'council_district', 'census_tract', 'bin', 'bbl', 'nta', 'location']


### Step 2: Convert data types
Convert dates to datetime and score/coordinates to numeric. The next cell verifies that conversion did not create new missing values.

In [3]:
def convert_types(df):
    """Convert text columns to proper dates and numbers."""
    df = df.copy()

    for col in ["inspection_date", "grade_date"]:
        df[col] = pd.to_datetime(df[col], format="%m/%d/%Y", errors="coerce")

    for col in ["score", "latitude", "longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["zipcode"] = df["zipcode"].str.strip()
    return df


df_typed = convert_types(df_named)
print(df_typed[["inspection_date", "grade_date", "score", "latitude", "longitude"]].dtypes)

inspection_date    datetime64[ns]
grade_date         datetime64[ns]
score                     float64
latitude                  float64
longitude                 float64
dtype: object


In [4]:
# Validation: conversion should not create new missing values
for col in ["inspection_date", "grade_date", "score", "latitude", "longitude"]:
    before = df_named[col].isna().sum()
    after = df_typed[col].isna().sum()
    status = "OK" if before == after else "CHECK"
    print(f"{col:<16} missing before: {before:>7}   after: {after:>7}   {status}")

inspection_date  missing before:       0   after:       0   OK
grade_date       missing before:  160401   after:  160401   OK
score            missing before:   17389   after:   17389   OK
latitude         missing before:    1666   after:    1666   OK
longitude        missing before:    1666   after:    1666   OK


### Step 3: Fix disguised missing values (Issues #3, #4, #14)
Replace placeholder values that actually mean "missing" with NaN: borough `"0"` and coordinates `(0, 0)`.

In [5]:
import numpy as np

def fix_disguised_missing(df):
    """Replace placeholder values that actually mean 'missing' with real NaN."""
    df = df.copy()

    # Borough "0" is not a valid NYC borough
    df["boro"] = df["boro"].replace("0", np.nan)

    # Coordinates (0, 0) are not in NYC
    zero_coords = (df["latitude"] == 0) | (df["longitude"] == 0)
    df.loc[zero_coords, ["latitude", "longitude"]] = np.nan

    return df


df_fixed = fix_disguised_missing(df_typed)

In [6]:
# Validation 1: borough
print("boro = '0' before:", (df_typed["boro"] == "0").sum())
print("boro = '0' after: ", (df_fixed["boro"] == "0").sum())
print("boro missing after:", df_fixed["boro"].isna().sum())

# Validation 2: coordinates
zero_before = ((df_typed["latitude"] == 0) | (df_typed["longitude"] == 0)).sum()
missing_before = df_typed["latitude"].isna().sum()
missing_after = df_fixed["latitude"].isna().sum()

print("\nzero coordinates before:", zero_before)
print("expected missing after:", missing_before + zero_before)
print("actual missing after:  ", missing_after)
print("Match:", missing_before + zero_before == missing_after)

boro = '0' before: 311
boro = '0' after:  0
boro missing after: 311

zero coordinates before: 3085
expected missing after: 4751
actual missing after:   4751
Match: True


### Step 4: Fill missing borough (Issues #5, #15, #16)
Fill missing borough from (1) the same restaurant's other rows, then (2) the most common borough for the same ZIP code. Filled rows are flagged in `boro_imputed`.

In [7]:
def fill_borough(df):
    """Fill missing borough using (1) the same restaurant's other rows, then (2) the most common borough for the same ZIP code."""
    df = df.copy()
    originally_missing = df["boro"].isna()

    # Step 1: borrow from the same restaurant
    boro_by_camis = (
        df.dropna(subset=["boro"])
        .groupby("camis")["boro"]
        .agg(lambda s: s.mode().iloc[0])
    )
    df["boro"] = df["boro"].fillna(df["camis"].map(boro_by_camis))

    # Step 2: borrow from restaurants in the same ZIP code
    boro_by_zip = (
        df.dropna(subset=["boro", "zipcode"])
        .groupby("zipcode")["boro"]
        .agg(lambda s: s.mode().iloc[0])
    )
    df["boro"] = df["boro"].fillna(df["zipcode"].map(boro_by_zip))

    # Flag rows that were missing and successfully filled
    df["boro_imputed"] = originally_missing & df["boro"].notna()

    return df


df_boro = fill_borough(df_fixed)

In [8]:
print("boro missing before:", df_fixed["boro"].isna().sum())
print("boro missing after: ", df_boro["boro"].isna().sum())
print("rows imputed:", df_boro["boro_imputed"].sum())

print("\n", df_boro["boro"].value_counts(dropna=False))

# Consistency check: each restaurant should belong to exactly one borough
boro_per_camis = df_boro.groupby("camis")["boro"].nunique()
print("\nrestaurants with more than one borough:", (boro_per_camis > 1).sum())

boro missing before: 311
boro missing after:  268
rows imputed: 43

 boro
Manhattan        109009
Brooklyn          74430
Queens            73836
Bronx             27418
Staten Island      9445
NaN                 268
Name: count, dtype: int64

restaurants with more than one borough: 0


In [9]:
# Why couldn't these be filled?
still_missing = df_boro["boro"].isna()
print("still missing:", still_missing.sum())
print("  of which also missing zipcode:", df_boro.loc[still_missing, "zipcode"].isna().sum())
print("  unique restaurants affected:", df_boro.loc[still_missing, "camis"].nunique())

still missing: 268
  of which also missing zipcode: 69
  unique restaurants affected: 96


In [10]:
# Chains: one brand name (dba) can have many locations (camis)
locations_per_name = df_boro.groupby("dba")["camis"].nunique().sort_values(ascending=False)
print(locations_per_name.head(10))

# Each camis should map to exactly one address
address_per_camis = df_boro.groupby("camis")[["building", "street"]].nunique()
print("\ncamis with more than one address:", ((address_per_camis > 1).any(axis=1)).sum())

dba
DUNKIN                    333
STARBUCKS                 168
MCDONALD'S                166
SUBWAY                    162
POPEYES                   121
DUNKIN'                   111
DOMINO'S                   73
CHIPOTLE MEXICAN GRILL     72
TACO BELL                  70
KENNEDY FRIED CHICKEN      68
Name: camis, dtype: int64

camis with more than one address: 0


### Step 5: Add inspection fields and analysis scope (Issues #2, #8, #18, #19)
Split `inspection_type` into program and stage, flag never-inspected restaurants (placeholder date 1900-01-01), and mark in-scope inspections in `in_scope`.

In [11]:
SCOPE_PROGRAMS = ["Cycle Inspection", "Pre-permit (Operational)"]
SCOPE_STAGES = ["Initial Inspection", "Re-inspection"]

def add_inspection_fields(df):
    """Split inspection type, flag never-inspected restaurants, and mark in-scope inspections."""
    df = df.copy()

    # Split "Program / Stage" into two columns
    split = df["inspection_type"].str.split(" / ", n=1, expand=True)
    df["inspection_program"] = split[0]
    df["inspection_stage"] = split[1]

    # Flag restaurants that have never been inspected (placeholder date)
    df["never_inspected"] = df["inspection_date"] == pd.Timestamp("1900-01-01")

    # Mark inspections within our analysis scope
    df["in_scope"] = (
        df["inspection_program"].isin(SCOPE_PROGRAMS)
        & df["inspection_stage"].isin(SCOPE_STAGES)
        & ~df["never_inspected"]
    )
    return df


df_insp = add_inspection_fields(df_boro)

In [12]:
# Check 1: never-inspected count should be 3,917
print("never inspected rows:", df_insp["never_inspected"].sum())

# Check 2: every inspection_type should split into exactly two parts
bad_split = df_insp["inspection_type"].notna() & df_insp["inspection_stage"].isna()
print("rows that failed to split:", bad_split.sum())

# Check 3: what does the split look like?
print("\n", df_insp["inspection_program"].value_counts(dropna=False))
print("\n", df_insp["inspection_stage"].value_counts(dropna=False))

# Check 4: size of the analysis scope
print("\nin-scope rows:", df_insp["in_scope"].sum())
print("in-scope inspections:",
      df_insp.loc[df_insp["in_scope"], ["camis", "inspection_date", "inspection_type"]].drop_duplicates().shape[0])

never inspected rows: 3917
rows that failed to split: 0

 inspection_program
Cycle Inspection                  217956
Pre-permit (Operational)           53616
Administrative Miscellaneous       11574
Pre-permit (Non-operational)        4005
NaN                                 3917
Inter-Agency Task Force             1397
Smoke-Free Air Act                   728
Calorie Posting                      576
Trans Fat                            503
Sodium Warning                        91
Accelerated Inspection Program        43
Name: count, dtype: int64

 inspection_stage
Initial Inspection              213426
Re-inspection                    69626
NaN                               3917
Compliance Inspection             3730
Reopening Inspection              3412
Second Compliance Inspection       295
Name: count, dtype: int64

in-scope rows: 264489
in-scope inspections: 77125


In [13]:
# Exact duplicate rows
print("exact duplicate rows:", df_insp.duplicated().sum())

# Inspection-level consistency: one inspection should have one score, one grade, one action
key = ["camis", "inspection_date", "inspection_type"]
in_scope = df_insp[df_insp["in_scope"]]

per_insp = in_scope.groupby(key).agg(
    n_scores=("score", "nunique"),
    n_grades=("grade", "nunique"),
    n_actions=("action", "nunique"),
)
print("\ninspections with more than one score: ", (per_insp["n_scores"] > 1).sum())
print("inspections with more than one grade: ", (per_insp["n_grades"] > 1).sum())
print("inspections with more than one action:", (per_insp["n_actions"] > 1).sum())

# Violation-level: is the same violation code listed twice in one inspection?
repeated_codes = in_scope.duplicated(subset=key + ["violation_code"]).sum()
print("\nrepeated violation codes within an inspection:", repeated_codes)

exact duplicate rows: 217

inspections with more than one score:  0
inspections with more than one grade:  0
inspections with more than one action: 0

repeated violation codes within an inspection: 206


In [14]:
# Remove exact duplicates first, then check again
in_scope_dedup = in_scope.drop_duplicates()
print("in-scope rows before:", len(in_scope))
print("in-scope rows after removing exact duplicates:", len(in_scope_dedup))
print("rows removed:", len(in_scope) - len(in_scope_dedup))

remaining = in_scope_dedup.duplicated(subset=key + ["violation_code"]).sum()
print("\nrepeated violation codes remaining:", remaining)

in-scope rows before: 264489
in-scope rows after removing exact duplicates: 264283
rows removed: 206

repeated violation codes remaining: 0


### Steps 6–7: Remove exact duplicates and flag grade/score mismatches (Issues #13, #20, #22)
Drop rows that are exact copies (no distinguishing information), then flag A/B/C grades that do not match the score band in `grade_score_mismatch`.

In [15]:
def remove_exact_duplicates(df):
    """Drop rows that are exact copies of another row."""
    before = len(df)
    df = df.drop_duplicates().copy()
    print(f"remove_exact_duplicates: dropped {before - len(df)} rows")
    return df


def flag_grade_score_mismatch(df):
    """Flag rows where an A/B/C grade does not match the score band."""
    df = df.copy()
    expected = pd.cut(df["score"], bins=[-1, 13, 27, 1000], labels=["A", "B", "C"]).astype(str)
    df["grade_score_mismatch"] = df["grade"].isin(["A", "B", "C"]) & (df["grade"] != expected)
    return df


df_dedup = remove_exact_duplicates(df_insp)
df_clean = flag_grade_score_mismatch(df_dedup)

remove_exact_duplicates: dropped 217 rows


In [16]:
# Check 1: dropped rows should be 217 (all data, not just in scope)
print("rows after dedup:", len(df_clean))

# Check 2: mismatch at inspection level, in scope
mismatch_insp = (
    df_clean.loc[df_clean["in_scope"] & df_clean["grade_score_mismatch"], key]
    .drop_duplicates()
    .shape[0]
)
print("in-scope inspections with grade/score mismatch:", mismatch_insp)

rows after dedup: 294189
in-scope inspections with grade/score mismatch: 28


## Build Tables
Split `df_clean` into three related tables:
- **restaurants**: one row per restaurant (primary key: `camis`)
- **inspections**: one row per inspection (primary key: `inspection_id`, foreign key: `camis`)
- **violations**: one row per violation (foreign key: `inspection_id`)

### Table 1: restaurants

Check whether restaurant attributes (name, cuisine, phone, ZIP) ever change across rows, to decide which value to keep.

In [17]:
# Do restaurant attributes ever change across rows?
for col in ["dba", "cuisine_description", "phone", "zipcode"]:
    n_changed = (df_clean.groupby("camis")[col].nunique() > 1).sum()
    print(f"{col:<22} restaurants with more than one value: {n_changed}")

dba                    restaurants with more than one value: 0
cuisine_description    restaurants with more than one value: 0
phone                  restaurants with more than one value: 0
zipcode                restaurants with more than one value: 0


### Step 8: Build restaurants table (Issues #23, #24)
One row per restaurant (`camis`), using attributes from its most recent record. A restaurant is flagged `never_inspected` only if all of its rows are placeholder rows.

In [18]:
RESTAURANT_COLS = [
    "camis", "dba", "boro", "building", "street", "zipcode", "phone",
    "cuisine_description", "latitude", "longitude",
    "community_board", "council_district", "census_tract", "nta",
    "boro_imputed",
]

def build_restaurants(df):
    """One row per restaurant (camis), using attributes from its most recent record."""
    cols = [c for c in RESTAURANT_COLS if c in df.columns]

    restaurants = (
        df.sort_values("inspection_date")
        .drop_duplicates(subset="camis", keep="last")[cols]
        .reset_index(drop=True)
    )

    # A restaurant is "never inspected" if ALL of its rows are placeholder rows
    never = df.groupby("camis")["never_inspected"].all()
    restaurants["never_inspected"] = restaurants["camis"].map(never)

    return restaurants


restaurants = build_restaurants(df_clean)
print(restaurants.shape)
restaurants.head()

(31369, 16)


,camis,dba,boro,building,street,zipcode,phone,cuisine_description,latitude,longitude,community_board,council_district,census_tract,nta,boro_imputed,never_inspected
0,50183597,98K CHICKEN & BURGERS,Brooklyn,1710,EAST 16 STREET,11229,6465689299,NaN,40.607812,-73.957168,315,48,055200,BK44,False,True
1,50186438,ESS-A-BAGEL,Manhattan,10,MURRAY STREET,10007,3477120094,NaN,40.713290,-74.007773,101,01,002100,MN24,False,True
2,50180719,"E & E NOODLES, INC.",Manhattan,42,WEST 38 STREET,10018,6468527672,NaN,40.751546,-73.984152,105,04,008400,MN17,False,True
3,50185873,SUYO MIO,Brooklyn,560,5 AVENUE,11215,8606147994,NaN,40.665262,-73.989572,307,39,014100,BK37,False,True
4,50153075,AAA+ CORP.,Brooklyn,2866,WEST 15 STREET,11224,9178178585,NaN,40.577725,-73.982829,313,47,034800,BK21,False,True


In [19]:
# Check 1: one row per restaurant
print("rows:", len(restaurants))
print("unique camis in df_clean:", df_clean["camis"].nunique())
print("camis is unique (primary key):", restaurants["camis"].is_unique)

# Check 2: never-inspected restaurants
print("\nnever-inspected restaurants:", restaurants["never_inspected"].sum())

# Check 3: is missing cuisine related to never being inspected?
print("\n", pd.crosstab(
    restaurants["never_inspected"],
    restaurants["cuisine_description"].isna(),
    rownames=["never_inspected"],
    colnames=["cuisine_missing"],
))

rows: 31369
unique camis in df_clean: 31369
camis is unique (primary key): True

never-inspected restaurants: 3917

 cuisine_missing  False  True 
never_inspected              
False            27450      2
True                 0   3917


### Step 9: Build inspections table
One row per inspection, excluding never-inspected placeholder rows. Adds violation counts (`n_violations`, `n_critical`) and a surrogate key `inspection_id`.

In [20]:
INSPECTION_KEY = ["camis", "inspection_date", "inspection_type"]

INSPECTION_COLS = INSPECTION_KEY + [
    "inspection_program", "inspection_stage", "action",
    "score", "grade", "grade_date",
    "in_scope", "grade_score_mismatch",
]

def build_inspections(df):
    """One row per inspection with violation counts. Excludes never-inspected placeholder rows."""
    real = df[~df["never_inspected"]].copy()
    real["is_critical"] = real["critical_flag"] == "Critical"

    # Inspection-level attributes (identical across rows of the same inspection)
    inspections = real.drop_duplicates(subset=INSPECTION_KEY)[INSPECTION_COLS]

    # Violation counts per inspection
    counts = (
        real.groupby(INSPECTION_KEY)
        .agg(
            n_violations=("violation_code", "count"),
            n_critical=("is_critical", "sum"),
        )
        .reset_index()
    )
    inspections = inspections.merge(counts, on=INSPECTION_KEY, how="left")

    # Surrogate key: a simple ID for each inspection
    inspections = inspections.sort_values(["camis", "inspection_date"]).reset_index(drop=True)
    inspections.insert(0, "inspection_id", range(1, len(inspections) + 1))

    return inspections


inspections = build_inspections(df_clean)
print(inspections.shape)
inspections.head()

(92713, 14)


,inspection_id,camis,inspection_date,inspection_type,inspection_program,inspection_stage,action,score,grade,grade_date,in_scope,grade_score_mismatch,n_violations,n_critical
0,1,30075445,2023-08-01,Cycle Inspection / Initial Inspection,Cycle Inspection,Initial Inspection,Violations were cited in the following area(s).,38.0,NaN,NaT,True,False,3,2
1,2,30075445,2023-08-22,Cycle Inspection / Re-inspection,Cycle Inspection,Re-inspection,Violations were cited in the following area(s).,12.0,A,2023-08-22,True,False,3,1
2,3,30075445,2024-11-08,Cycle Inspection / Initial Inspection,Cycle Inspection,Initial Inspection,Violations were cited in the following area(s).,10.0,A,2024-11-08,True,False,3,1
3,4,30075445,2026-02-27,Cycle Inspection / Initial Inspection,Cycle Inspection,Initial Inspection,Violations were cited in the following area(s).,7.0,A,2026-02-27,True,False,2,1
4,5,30191841,2024-11-20,Cycle Inspection / Initial Inspection,Cycle Inspection,Initial Inspection,Violations were cited in the following area(s).,24.0,NaN,NaT,True,False,6,2


In [21]:
# Check 1: primary key and inspection key are both unique
print("rows:", len(inspections))
print("inspection_id is unique:", inspections["inspection_id"].is_unique)
print("inspection key is unique:", not inspections.duplicated(subset=INSPECTION_KEY).any())

# Check 2: in-scope inspections should match earlier count
print("\nin-scope inspections:", inspections["in_scope"].sum())

# Check 3: reconciliation - total violations should equal non-null violation codes in the raw rows
real_rows = df_clean[~df_clean["never_inspected"]]
print("\nsum of n_violations:", inspections["n_violations"].sum())
print("non-null violation codes:", real_rows["violation_code"].notna().sum())

# Check 4: inspections with zero violations vs. "No violations" action
print("\n", pd.crosstab(
    inspections["n_violations"] == 0,
    inspections["action"].str.startswith("No violations", na=False),
    rownames=["zero_violations"],
    colnames=["action_no_violations"],
))

rows: 92713
inspection_id is unique: True
inspection key is unique: True

in-scope inspections: 77125

sum of n_violations: 287206
non-null violation codes: 287206

 action_no_violations  False  True 
zero_violations                   
False                 89484    163
True                    306   2760


Investigate inspections where the ACTION text conflicts with the recorded violation count (Issue #25). Decision: use `n_violations` as the source of truth.

In [22]:
# Exception A: zero violations, but action is not "No violations"
mask_a = (inspections["n_violations"] == 0) & ~inspections["action"].str.startswith("No violations", na=False)
print("Exception A by action:")
print(inspections.loc[mask_a, "action"].value_counts())
print("in scope:", inspections.loc[mask_a, "in_scope"].sum())

# Exception B: "No violations" action, but violations recorded
mask_b = (inspections["n_violations"] > 0) & inspections["action"].str.startswith("No violations", na=False)
print("\nException B by program:")
print(inspections.loc[mask_b, "inspection_program"].value_counts())
print("in scope:", inspections.loc[mask_b, "in_scope"].sum())

Exception A by action:
action
Establishment re-opened by DOHMH.                                                                                                     298
Violations were cited in the following area(s).                                                                                         5
Establishment Closed by DOHMH. Violations were cited in the following area(s) and those requiring immediate action were addressed.      2
Establishment re-closed by DOHMH.                                                                                                       1
Name: count, dtype: int64
in scope: 4

Exception B by program:
inspection_program
Administrative Miscellaneous    148
Cycle Inspection                  6
Smoke-Free Air Act                3
Pre-permit (Operational)          3
Trans Fat                         2
Calorie Posting                   1
Name: count, dtype: int64
in scope: 9


### Step 10: Build violations table (Issues #26, #27)
One row per violation, linked to its inspection via `inspection_id`. Adds `violation_category` from the first two characters of the violation code.

In [23]:
VIOLATION_COLS = ["violation_code", "violation_description", "critical_flag"]

def build_violations(df, inspections):
    """One row per violation, linked to its inspection via inspection_id."""
    real = df[~df["never_inspected"] & df["violation_code"].notna()]

    # Attach inspection_id by matching on the inspection key
    violations = real[INSPECTION_KEY + VIOLATION_COLS].merge(
        inspections[INSPECTION_KEY + ["inspection_id"]],
        on=INSPECTION_KEY,
        how="left",
        validate="many_to_one",
    )

    # Violation category = first two characters of the code (e.g., "04L" -> "04")
    violations["violation_category"] = violations["violation_code"].str[:2]

    violations = violations[["inspection_id"] + VIOLATION_COLS + ["violation_category"]]
    violations = violations.sort_values(["inspection_id", "violation_code"]).reset_index(drop=True)
    violations.insert(0, "violation_id", range(1, len(violations) + 1))

    return violations


violations = build_violations(df_clean, inspections)
print(violations.shape)
violations.head()

(287206, 6)


,violation_id,inspection_id,violation_code,violation_description,critical_flag,violation_category
0,1,1,04L,Evidence of mice or live mice in establishment...,Critical,04
1,2,1,05H,No approved written standard operating procedu...,Critical,05
2,3,1,08A,Establishment is not free of harborage or cond...,Not Critical,08
3,4,2,04L,Evidence of mice or live mice in establishment...,Critical,04
4,5,2,08A,Establishment is not free of harborage or cond...,Not Critical,08


In [24]:
# Check 1: row count should match the reconciliation total
print("rows:", len(violations))
print("violation_id is unique:", violations["violation_id"].is_unique)

# Check 2: every violation is linked to an inspection (foreign key integrity)
print("\nviolations without inspection_id:", violations["inspection_id"].isna().sum())
print("all inspection_ids exist in inspections table:",
      violations["inspection_id"].isin(inspections["inspection_id"]).all())

# Check 3: no repeated code within an inspection
print("\nrepeated code within an inspection:",
      violations.duplicated(subset=["inspection_id", "violation_code"]).sum())

# Check 4: critical counts reconcile with the inspections table
print("\ncritical violations in violations table:", (violations["critical_flag"] == "Critical").sum())
print("sum of n_critical in inspections table: ", inspections["n_critical"].sum())

# Check 5: most common violation categories
print("\n", violations["violation_category"].value_counts().head(10))

rows: 287206
violation_id is unique: True

violations without inspection_id: 0
all inspection_ids exist in inspections table: True

repeated code within an inspection: 0

critical violations in violations table: 153861
sum of n_critical in inspections table:  153861

 violation_category
10    76428
06    54060
04    51111
02    37782
08    31732
09    10376
05     8584
20     7207
28     3744
03     2015
Name: count, dtype: int64


Show the most common description for each violation category to interpret what each code prefix means (Issue #28).

In [25]:
# What does each category mean? Show the most common description per category
category_lookup = (
    violations.groupby("violation_category")["violation_description"]
    .agg(lambda s: s.value_counts().index[0])
)
for cat, desc in category_lookup.items():
    print(f"{cat}: {desc[:90]}")

02: Cold TCS food item held above 41 °F; smoked or processed fish held above 38 °F; intact raw
03: Food, prohibited, from unapproved or unknown source, home canned or home prepared. Animal 
04: Evidence of mice or live mice in establishment's food or non-food areas.
05: No hand washing facility in or adjacent to toilet room or within 25 feet of a food prepara
06: Food contact surface not properly washed, rinsed and sanitized after each use and followin
07: Duties of an officer of the Department interfered with or obstructed.
08: Establishment is not free of harborage or conditions conducive to rodents, insects or othe
09: Design, construction, materials used or maintenance of food contact surface improper. Surf
10: Non-food contact surface or equipment made of unacceptable material, not kept clean, or no
15: Sign prohibiting smoking or using electronic cigarettes not conspicuously posted.
16: The original nutrition fact labels or ingredient label for a cooking oil, shortening or ma
18:

### Save tables
Write all three tables to `data/processed/`, then re-read each file to confirm row counts match.

In [26]:
restaurants.to_csv(PROCESSED_DIR + "restaurants.csv", index=False)
inspections.to_csv(PROCESSED_DIR + "inspections.csv", index=False)
violations.to_csv(PROCESSED_DIR + "violations.csv", index=False)

# Verify: re-read each file and compare row counts
for name, table in [("restaurants", restaurants), ("inspections", inspections), ("violations", violations)]:
    reloaded = pd.read_csv(PROCESSED_DIR + f"{name}.csv")
    status = "OK" if len(reloaded) == len(table) else "CHECK"
    print(f"{name:<12} saved: {len(table):>7}   reloaded: {len(reloaded):>7}   {status}")

restaurants  saved:   31369   reloaded:   31369   OK
inspections  saved:   92713   reloaded:   92713   OK
violations   saved:  287206   reloaded:  287206   OK


## Assemble the Pipeline
Chain all cleaning and table-building steps into one function, and convert key integrity checks into `assert` statements that stop with an error if a check fails. This is the foundation for the reusable pipeline in Phase 2.

In [27]:
def run_cleaning(raw):
    """Run all cleaning and table-building steps in order."""
    df = standardize_columns(raw)
    df = convert_types(df)
    df = fix_disguised_missing(df)
    df = fill_borough(df)
    df = add_inspection_fields(df)
    df = remove_exact_duplicates(df)
    df = flag_grade_score_mismatch(df)

    restaurants = build_restaurants(df)
    inspections = build_inspections(df)
    violations = build_violations(df, inspections)
    return restaurants, inspections, violations


def validate_tables(restaurants, inspections, violations):
    """Stop with an error if any integrity check fails."""
    assert restaurants["camis"].is_unique, "restaurants: camis is not unique"
    assert inspections["inspection_id"].is_unique, "inspections: inspection_id is not unique"
    assert violations["inspection_id"].isin(inspections["inspection_id"]).all(), "violations: orphan records found"
    assert inspections["n_violations"].sum() == len(violations), "violation counts do not reconcile"
    assert inspections["n_critical"].sum() == (violations["critical_flag"] == "Critical").sum(), "critical counts do not reconcile"
    print("All validation checks passed.")

In [28]:
# Run the assembled pipeline and confirm it reproduces the step-by-step results
r_pipe, i_pipe, v_pipe = run_cleaning(raw)
validate_tables(r_pipe, i_pipe, v_pipe)

print("\nrestaurants identical:", r_pipe.equals(restaurants))
print("inspections identical:", i_pipe.equals(inspections))
print("violations identical: ", v_pipe.equals(violations))

remove_exact_duplicates: dropped 217 rows
All validation checks passed.

restaurants identical: True
inspections identical: True
violations identical:  True
